# Aggregation + Statistical Hypothesis Pipeline — Demo (input_v2)

End-to-end run of `run_aggregation_and_hypothesis_pipeline.py` against
`hypothesis_framework/data/input_v2` with ground truth from
`hypothesis_framework/data/groundtruth/kubernetes`.

**What this notebook does**

1. Loads `.env` (Azure OpenAI credentials required by the LLM Council).
2. Stages `input_v2/**/*.json` into `input_v2_metrics/` with the
   `*_metrics.json` suffix the aggregator's `DirectoryQueryService` expects.
3. Runs the pipeline with `--advanced-analysis` so the merged JSON
   contains the `statistical_hypothesis` key.
4. Inspects the final consolidated JSON.


## 1. Setup — repo root & .env

In [ ]:
import os, sys, shutil, json, subprocess
from pathlib import Path

REPO_ROOT = Path(r"C:\Users\meemankgupta\Music\Project\infosys\certifier")
os.chdir(REPO_ROOT)
print("CWD:", Path.cwd())

env_file = REPO_ROOT / ".env"
if env_file.exists():
    for line in env_file.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        os.environ[k.strip()] = v.strip().strip('"').strip("'")
    print("Loaded .env keys:", [k for k in os.environ if k.startswith(("AZURE_", "ENV_", "MONGODB"))])
else:
    print("WARNING: .env not found at", env_file)


## 2. Stage `input_v2` → `input_v2_metrics` (rename to `*_metrics.json`)

In [ ]:
SRC = REPO_ROOT / "hypothesis_framework" / "data" / "input_v2"
DST = REPO_ROOT / "hypothesis_framework" / "data" / "input_v2_metrics"

if DST.exists():
    shutil.rmtree(DST)
DST.mkdir(parents=True)

count = 0
for cat_dir in sorted(SRC.iterdir()):
    if not cat_dir.is_dir():
        continue
    out_cat = DST / cat_dir.name
    out_cat.mkdir(parents=True, exist_ok=True)
    for f in cat_dir.glob("*.json"):
        shutil.copy(f, out_cat / f"{f.stem}_metrics.json")
        count += 1

print(f"Staged {count} files into {DST}")
for d in sorted(DST.iterdir()):
    if d.is_dir():
        print(f"  {d.name}: {len(list(d.glob('*.json')))} files")


## 3. Run the pipeline (aggregation + hypothesis)

In [ ]:
METRICS_DIR     = "hypothesis_framework/data/input_v2_metrics"
OUTPUT_DIR      = "hypothesis_framework/data/output_v2"
GROUND_TRUTH    = "hypothesis_framework/data/groundtruth/kubernetes"
AGENT_ID        = "sre-agent-v2.1"
AGENT_NAME      = "SRE-Agent"
RUNS_PER_FAULT  = 30
MIN_RUNS        = 30

cmd = [
    sys.executable, "run_aggregation_and_hypothesis_pipeline.py",
    "--metrics-dir",       METRICS_DIR,
    "--output-dir",        OUTPUT_DIR,
    "--agent-id",          AGENT_ID,
    "--agent-name",        AGENT_NAME,
    "--advanced-analysis",
    "--ground-truth-dir",  GROUND_TRUTH,
    "--runs-per-fault",    str(RUNS_PER_FAULT),
    "--min-runs",          str(MIN_RUNS),
]

print("Running:")
print(" ", " ".join(cmd))
print()

result = subprocess.run(cmd, capture_output=True, text=True)
print("--- STDOUT (tail) ---")
print("\n".join(result.stdout.splitlines()[-25:]))
print("\n--- STDERR (tail) ---")
print("\n".join(result.stderr.splitlines()[-15:]))
print("\nExit code:", result.returncode)


## 4. Inspect the consolidated output

In [ ]:
out_path = Path(OUTPUT_DIR) / f"aggregated_with_hypothesis_{AGENT_ID}.json"
print("Output path:", out_path, "  size:", round(out_path.stat().st_size / 1024, 1), "KB")

data = json.loads(out_path.read_text(encoding="utf-8"))

print("\nTop-level keys:")
for k in data.keys():
    print(f"  - {k}")

print(f"\nagent_name              : {data.get('agent_name')}")
print(f"total_runs              : {data.get('total_runs')}")
print(f"total_faults_tested     : {data.get('total_faults_tested')}")
print(f"total_fault_categories  : {data.get('total_fault_categories')}")
print(f"runs_per_fault          : {data.get('runs_per_fault')}")
print(f"statistical_hypothesis ?: {'statistical_hypothesis' in data}")


### 4a. Per-category quick view

In [ ]:
for sc in data.get("fault_category_scorecards", []):
    derived = sc.get("derived_metrics", {})
    num     = sc.get("numeric_metrics", {})
    print(f"\n• {sc['fault_category']}  ({sc['total_runs']} runs)")
    print(f"    sub-faults             : {', '.join(sc.get('faults_tested', []))}")
    print(f"    detection success rate : {derived.get('fault_detection_success_rate')}")
    print(f"    mitigation success rate: {derived.get('fault_mitigation_success_rate')}")
    print(f"    RAI compliance rate    : {derived.get('rai_compliance_rate')}")
    print(f"    security compliance    : {derived.get('security_compliance_rate')}")
    ttd = num.get("time_to_detect", {}).get("median")
    ttm = num.get("time_to_mitigate", {}).get("median")
    if ttd is not None: print(f"    TTD (median)           : {ttd}s")
    if ttm is not None: print(f"    TTM (median)           : {ttm}s")


### 4b. statistical_hypothesis block

In [ ]:
sh = data.get("statistical_hypothesis", {})
print("metadata keys :", list(sh.get("metadata", {}).keys()))
print("validation    :", {k: sh.get("validation", {}).get(k) for k in ("total_runs","total_detected","status")})
print("hypothesis IDs:", list(sh.get("results", {}).keys()))
print("\nExample - H01 metrics covered:")
for metric, res in sh.get("results", {}).get("h01", {}).items():
    status = res.get("status", "ok")
    print(f"  {metric:<20}  status={status}")


## 5. (Optional) Cleanup staging directory

In [ ]:
# Uncomment to remove the staged metrics directory after the run
# shutil.rmtree(DST, ignore_errors=True)
# print('Removed', DST)
